# Notebook 01: DeepSAM: the model, the master equation, and the learned surplus

**Course:** Summer School on AI for Economics and Finance · ESOMAS, University of Torino (August 24–26, 2026)
**Session:** Day 2, 11:00 – 12:30: Deep Learning for Continuous-Time Models
**Slides:** `../../slides/HACT_DeepSAM_Lecture_Slides.pdf`
**Notebook role:** core (in-class walkthrough)
**Runtime:** ~1 min at `smoke`, ~3 min at `production`
**Created by:** Yucheng Yang. [Course repository](https://github.com/yangycpku/summer-school-AI-for-economics-and-finance-2026)

---


This notebook is Section 3 of *Deep Learning for Search and Matching Models*: a labour
search-and-matching economy with **two-sided heterogeneity** (worker types $x$, firm types
$y$), **aggregate shocks**, and **distributional feedback** — the distribution of existing
matches changes the value of forming new ones.

The theory below is the write-up from the replication package, unchanged. The code then
does three things:

1. solves the deterministic steady state for each aggregate state;
2. loads the trained surplus network;
3. measures **how well the master equation is actually satisfied**, type by type — which is
   the only honest way to judge a solution when there is no analytic benchmark.

In [ ]:
RUN_MODE = "smoke"     # one of: "smoke", "teaching", "production"

## Locating the code

`src/train_nn.py` holds the whole method: the deterministic steady states, the neural
networks, the master-equation residual, the simulation of the distribution, and the training
loop. It expects to be imported with the project root as the working directory, because
`solve_steady_state` writes its output there as `.npy` files.

On Nuvolos the notebook server starts in `/files`, which mirrors the course repository, so
the cell below changes into `/files/day2/Yang/code/DeepSAM_nuvolos`. On a local clone or
Colab it steps up from `notebooks/` to the project root instead.

In [ ]:
import os
import sys
from pathlib import Path

# On Nuvolos the kernel starts in /files, which mirrors the course repository.
NUVOLOS_ROOT = "/files/day2/Yang/code/DeepSAM_nuvolos"
if os.path.isdir(NUVOLOS_ROOT):
    os.chdir(NUVOLOS_ROOT)
elif os.path.isfile("../src/train_nn.py"):
    os.chdir("..")           # a local clone or Colab: the notebook lives in notebooks/

if not (os.path.isfile("src/train_nn.py") and os.path.isfile("config/config.yaml")):
    raise FileNotFoundError(
        f"Expected the DeepSAM project root, but the working directory is "
        f"{os.getcwd()!r}. On Nuvolos that is {NUVOLOS_ROOT!r}; elsewhere, open this "
        f"notebook from inside DeepSAM_nuvolos/notebooks."
    )

ROOT = Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

In [ ]:
import contextlib
import io
import random
import time

import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from omegaconf import OmegaConf

from train_nn import Train_NN, Master_PINN_S
import calibration_plot as calplot
import covid_shock_plot as covplot
import plotting

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
else:
    print("CPU (no GPU visible) -- everything below still runs, more slowly")

## Choosing the run mode

The costs in this notebook are all *simulation* costs: how many paths of the economy are
simulated, and for how long. `RUN_MODE` maps onto them.

| | `smoke` | `teaching` | `production` |
|---|---|---|---|
| ergodic-pool paths × horizon | 32 × 500 | 64 × 1000 | 256 × 5000 |
| pre-COVID ergodic paths | 50 | 100 | 200 |
| recovery paths averaged | 20 | 60 | 200 |
| training steps (notebook 03) | 500 | 5,000 | 20,000 |

`production` matches the settings in the replication package. Note what is *not* on this
list: training the surplus network to convergence. The full pipeline behind the shipped
checkpoint is a homotopy initialisation, a long main training phase run to a loss
threshold, and then 8 further rounds of 100,000 gradient steps with the ergodic dataset
rebuilt between rounds — several hours on an A100. That is why every notebook here starts
from the shipped checkpoint, and why notebook 03 quantifies the gap rather than trying to
close it.

In [ ]:
if RUN_MODE == "smoke":
    SIM_PATHS, SIM_T = 32, 500
    ERG_PATHS, ERG_T_END = 50, 10.0
    RECOVERY_PATHS = 20
    TRAIN_STEPS = 500
elif RUN_MODE == "teaching":
    SIM_PATHS, SIM_T = 64, 1000
    ERG_PATHS, ERG_T_END = 100, 20.0
    RECOVERY_PATHS = 60
    TRAIN_STEPS = 5_000
elif RUN_MODE == "production":
    SIM_PATHS, SIM_T = 256, 5000
    ERG_PATHS, ERG_T_END = 200, 30.0
    RECOVERY_PATHS = 200
    TRAIN_STEPS = 20_000
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

print(
    f"RUN_MODE={RUN_MODE}: ergodic pool {SIM_PATHS}x{SIM_T}, "
    f"{ERG_PATHS} pre-COVID paths, {RECOVERY_PATHS} recovery paths"
)

## **I. Economic environment**

We work in continuous time with an infinite horizon. The economy is populated by heterogeneous workers and heterogeneous firms. A worker has type $x \in \mathcal X$, and a firm has type $y \in \mathcal Y$.

The aggregate state of the economy is denoted by $z_t \in \mathcal Z$. It follows a continuous-time Markov chain with transition intensities
$
\lambda(z,\check{z}).
$

In the Section 3 application, $\mathcal Z$ contains three aggregate states: an expansion state, a normal recession state, and a disaster state.

A worker can be either unemployed ($u$) or employed ($e$) in a match. A firm can be either vacant ($v$) or producing ($p$) in a match. If a worker of type $x$ is matched with a firm of type $y$, the match produces flow output
$
F(x,y,z).
$

An unmatched worker receives flow value $b$, while a vacant firm pays flow cost $c$. Matches are destroyed at the exogenous separation rate
$
\delta(x,y,z).
$

The central endogenous state variable is the match distribution

$$
g_t(x,y),
$$

which describes the mass of active matches between workers of type $x$ and firms of type $y$. From $g_t$, we obtain the mass of employed workers and producing firms:

$$
g_t^e(x) = \int_{\mathcal Y} g_t(x,y)dy,
\qquad
g_t^p(y) = \int_{\mathcal X} g_t(x,y)dx.
$$

Given the total worker distribution $g^w(x)$ and the total firm distribution $g_t^f(y)$, the unmatched worker and vacant firm distributions are

$$
g_t^u(x) = g^w(x) - g_t^e(x),
\qquad
g_t^v(y) = g_t^f(y) - g_t^p(y).
$$

Aggregate unemployment, aggregate employment, aggregate vacant firms, and aggregate producing firms are therefore

$$
U_t = \int_{\mathcal X} g_t^u(x)dx,
\qquad
E_t = \int_{\mathcal X} g_t^e(x)dx,
\qquad
V_t = \int_{\mathcal Y} g_t^v(y)dy,
\qquad
P_t = \int_{\mathcal Y} g_t^p(y)dy.
$$

Unmatched workers and vacant firms meet through a matching function

$$
m(U_t,V_t).
$$

The meeting rate for an unmatched worker and a vacant firm are

$$
M_t^u = \frac{m(U_t,V_t)}{U_t},
\qquad
M_t^v = \frac{m(U_t,V_t)}{V_t}.
$$
respectively.

When a worker and firm meet, they decide whether to form a match. This decision is summarized by the acceptance rule

$$
\alpha(x,y,z,g) \in [0,1].
$$

Thus, the state of the economy is not only the aggregate shock $z$, but the pair

$$
(z,g).
$$

That is, the distribution $g$ affects who is available to meet, which changes matching decisions, which in turn changes the future distribution.


## **II. Recursive characterization of equilibrium**

The recursive state is $(z,g)$, where $z$ is the aggregate state and $g$ is the match distribution.

Let $V^u$ and $V^e$ denote the values of unmatched and employed workers, and let $V^v$ and $V^p$ denote the values of vacant and producing firms. The match surplus is

$$
S(x,y,z,g)
:=
V^p(x,y,z,g)-V^v(y,z,g)
+
V^e(x,y,z,g)-V^u(x,z,g).
$$

Surplus $S(x,y,z,g)$ is divided between workers and firms according to Nash Bargaining protocol with the bargaining parameter $\beta$. Thus,

$$
\begin{aligned}
	\beta S(x,y,z,g) ={}& V^e(x,y,z,g) - V^u(x,z,g) \\
	(1-\beta) S(x,y,z,g) ={}& V^p(x,y,z,g) - V^v(y,z,g).
\end{aligned}
$$

A meeting is accepted when the surplus is positive. In the exact model,

$$
\alpha(x,y,z,g)=\mathbf 1_{S(x,y,z,g)\geq 0}.
$$

In the numerical implementation, we use the smooth approximation

$$
\alpha(x,y,z,g)
=
\frac{1}{1+e^{-\xi S(x,y,z,g)}} \qquad \Rightarrow \qquad \lim_{\xi\rightarrow \infty}\alpha(x,y,z,g)=\mathbf 1_{S(x,y,z,g)\geq 0} \ \ \forall \ \  S(x,y,z,g)\neq0.
$$

The match distribution evolves according to

$$dg_t(x,y) =\mu_t^g(x,y,z,g) dt - \sum_{\check{z}_t \ne z_t} \sigma(z_t, \check{z}_t) g_t(x,y) dN(z_t;\check{z}_t),$$
where
$$
\mu^g(x,y,z,g)
=
-\big(\delta(x,y,z)+\varsigma(z)\big)g(x,y)
+
\alpha(x,y,z,g)m(U,V)
\frac{g^u(x)}{U}
\frac{g^v(y)}{V}.
$$

The first term removes matches through separations and firm exit. The second term adds newly accepted matches. When the aggregate state jumps from $z$ to $\check{z}$, a fraction $\sigma(z,\check{z})$ of matches is destroyed, so the post-jump distribution is

$$\bigl(1-\sigma(z,\check{z})\bigr)g.$$

In equilibrium, agents correctly forecast the law of motion of $g$. This means that the drift and jump they use in their value functions must equal the equilibrium the drift $\mu^g$ and jump $-\sigma(z_t, \check{z}_t) g_t(x,y)$. Therefore, the recursive equilibrium can be characterized directly through the surplus function $S(x,y,z,g)$.




## **III. Agent Hamilton-Jacobi-Bellman (HJB) Equations**

**Workers:**
Given beliefs and optimal decisions, the worker value functions $V^u$ and $V^e$ satisfy the Hamilton Jacobi Bellman (HJB) Equations: 
\begin{align}
\rho V^u(x,z,g) 
	={}& b + M^u \int \alpha(x,\tilde{y},z,g) 
    (V^e(x,\tilde{y},z,g) - V^u(x,z,g))
    \frac{g^v(\tilde{y})}{V}  d\tilde{y} \nonumber \\
    {}& \hspace{-2.5cm} + \sum_{\check{z} \ne z} \lambda(z,\check{z})\left(V^u(x,\check{z}, (1-\sigma(z,\check{z}))g) - V^u(x,z,g)\right) + \langle D_{g} V^u, \breve{\mu}^g \rangle
    \\
%%
\rho V^e(x,y,z,g)
	={}&  \nonumber w(x,y,z,g) + \delta(x,y,z) (V^u(x,z,g) - V^e(x,y,z,g)) \nonumber \\
    {}& \hspace{-2.5cm} + \sum_{\check{z}\ne z} \lambda(z,\check{z}) \left(V^e(x, y, \check{z},(1-\sigma(z,\check{z}))g) - V^e(x,y,z,g)\right) + \langle D_{g} V^e, \breve{\mu}^g \rangle
% \rho V^e(x,y,z,g)
% 	={}&  \nonumber w(x,y,z,g) + (\delta(x,y,z) + \varsigma(z)) (V^u(x,z,g) - V^e(x,y,z,g)) \nonumber \\
%     {}& \hspace{-1.5cm} + \sum_{\check{z}\ne z} \lambda(z,\check{z})\big[(1-\sigma(z,\check{z}))\left(V^e(x, y, \check{z},(1-\sigma(z,\check{z}))g) - V^e(x,y,z,g)\right) \nonumber \\
%     {}& \hspace{-1.5cm} + \sigma(z,\check{z})\left(V^u(x,\check{z}, (1-\sigma(z,\check{z}))g) - V^e(x,y,z,g)\right)\big] + \langle D_{g} V^e, \breve{\mu}^g \rangle
%     \label{eq:general:hjbes:Ve}
\end{align}



**Firms:**
Given beliefs and optimal decisions, the firm value functions $V^v$ and $V^p$ satisfy the Hamilton Jacobi Bellman (HJB) Equations: 
\begin{align}
%%
\rho V^v(y,z,g)
	={}& - c + M^v \int \alpha(\tilde{x},y,z,g) 
    (V^p(\tilde{x},y,z,g) - V^v(y,z,g)) \frac{g^u(\tilde{x})}{U} d\tilde{x} \nonumber \\
    {}& \hspace{-2.5cm} + \sum_{\check{z} \ne z} \lambda(z,\check{z})\left(V^v(y,\check{z},(1-\sigma(z,\check{z}))g) - V^v(y,z,g)\right) + \left\langle D_{g} V^v, \breve{\mu}^g \right\rangle
    \\
%%
\rho V^p(x,y,z,g)
	={}& F(x,y,z) - w(x,y,z,g) + \delta(x,y,z) (V^v(y,z,g) - V^p(x,y,z,g)) \nonumber \\
    {}& \hspace{-2.5cm} + \sum_{\check{z} \ne z} \lambda(z,\check{z}) (V^p(x, y,\check{z},(1-\sigma(z,\check{z}))g) - V^p(x, y, z, g)) + \left\langle D_{g} V^p, \breve{\mu}^g \right\rangle 
\end{align}

## **IV. Free-entry condition**

Free entry requires the expected value of a vacancy to be zero:

$$
0
=
\mathbb E_{\widetilde y}
\left[
V^v(\widetilde y,z,g)
\right]
=
\int_0^1
V^v(\widetilde y,z,g)\,\bar h(\widetilde y)\,d\widetilde y .
$$

Combining free entry with the vacancy HJB equation and the surplus-sharing rule gives

$$
\frac{m(U_t,V_t)}{V_t}
=
\frac{
c
}{
\displaystyle
\int
\int
\alpha(\widetilde x,\widetilde y,z_t,g_t)
\frac{g_t^u(\widetilde x)}{U_t}
(1-\beta)
S(\widetilde x,\widetilde y,z_t,g_t)
\,d\widetilde x\,d\widetilde y
}.
$$

Since the matching function is homothetic, this equation pins down the vacancy mass $V_t$. With uniform firm entry draws, the total firm distribution is

$$
g_t^f(y)=V_t+P_t,
$$

where

$$
P_t=\int_{\mathcal Y}g_t^p(y)\,dy .
$$

## **V. Master equation**

Combining the worker and firm HJB equations gives one master equation for the surplus:

$$
\begin{aligned}
0 = \mathcal L^S_S :=\;&
-\rho S(x,y,z,g)
+ F(x,y,z)
- \delta(x,y,z)S(x,y,z,g)
- b
\\
&-
\beta \frac{m(U,V)}{U}
\int_{\mathcal Y}
\alpha(x,\widetilde y,z,g)
S(x,\widetilde y,z,g)
\frac{g^v(\widetilde y)}{V}
\,d\widetilde y
\\
&+
c
-
(1-\beta)\frac{m(U,V)}{V}
\int_{\mathcal X}
\alpha(\widetilde x,y,z,g)
S(\widetilde x,y,z,g)
\frac{g^u(\widetilde x)}{U}
\,d\widetilde x
\\
&+
\sum_{\widehat z \neq z}
\lambda(z,\widehat z)
\left[
S\!\left(x,y,\widehat z,(1-\sigma(z,\widehat z))g\right)
-
S(x,y,z,g)
\right]
\\
&+
\left\langle D_g S,\mu^g \right\rangle .
\end{aligned}
$$

Using the master equation, $\alpha$ definition, KFE, "free-entry" condition, and beliefs consistency, DeepSAM approximates $S(x,y,z,g)$ with a neural network. Once $S$ is known, we recover the acceptance rule $\alpha$ and simulate the evolution of $g$.

## Calibration and the deterministic steady states

The calibration and training settings all live in `config/config.yaml`. We load it with
OmegaConf and pass it straight to `Train_NN`, so it is easy to see that the object is
simply built from that dictionary.

`solve_steady_state()` then solves the model's **deterministic** steady state once for each
aggregate state $z \in \{L, H, D\}$ — low, high, and the disaster state that stands in for
COVID. These are fixed points of the matching problem with the aggregate state frozen; they
are the anchors the aggregate-risk solution is built around, and the unemployment rates they
imply are the first thing to sanity-check against the calibration.

In [ ]:
cfg = OmegaConf.load(ROOT / "config" / "config.yaml")
params = {
    k: v for k, v in OmegaConf.to_container(cfg.train_nn, resolve=True).items()
    if k != "_target_"
}

seed = int(cfg.seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ct = Train_NN(**params)
print(f"device {ct.device} | {ct.nx} worker types x {ct.ny} firm types | output path {ct.path}")

t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):     # the solver is chatty; keep the summary
    ct.solve_steady_state()
print(f"solve_steady_state: {time.monotonic() - t0:.1f}s")

gm_ss = np.load("gm_ss.npy")
gm_low = np.load("gm_low_delta.npy")
gm_high = np.load("gm_high_delta.npy")
gm_dis = np.load("gm_dis_delta.npy")

# State convention (see env.py): L is the good state (separation delta_0 - d_delta),
# H the bad state (delta_0 + d_delta), D the disaster state.
for label, gm in [("baseline", gm_ss), ("low separation (L)", gm_low),
                  ("high separation (H)", gm_high), ("disaster (D)", gm_dis)]:
    u = (ct.gw.mean() - np.mean(gm)).cpu().numpy() * 100
    print(f"  unemployment rate, {label:<22s}: {u:6.3f}%")

## The trained surplus network

The one object DeepSAM learns is the **match surplus** $S(x, y, z, g)$: the value of a match
between worker type $x$ and firm type $y$, given the aggregate state $z$ *and the entire
cross-sectional distribution* $g$ of existing matches. That last argument is the hard part —
$g$ lives in $\mathbb{R}^{n_x \times n_y}$ (55 dimensions here), which is why the network
takes it directly as an input rather than summarising it.

Everything else in the model is recovered from $S$: the acceptance sets, the vacancy
posting implied by free entry, the wage through Nash bargaining, and the drift of $g$
itself.

The checkpoint below is the converged network from the paper. Notebook 03 shows what
training it involves.

In [ ]:
pinn_S = Master_PINN_S(
    nn_width=ct.nn_width,
    nn_num_layers=ct.nn_num_layers,
    n_x=ct.nx,
    n_y=ct.ny,
).to(ct.device).float()

ckpt = torch.load(ROOT / "checkpoints" / "section3_surplus_best.pt", map_location=ct.device)
pinn_S.load_state_dict(ckpt["model_state_dict"])
pinn_S.eval()

n_par = sum(p.numel() for p in pinn_S.parameters())
print(f"Loaded the trained surplus network: {ct.nn_num_layers} layers of width "
      f"{ct.nn_width}, {n_par:,} parameters")
print(f"Input dimension: 1 (x) + 1 (y) + 1 (z) + {ct.nx * ct.ny} (g) = {3 + ct.nx * ct.ny}")

## Simulating the ergodic distribution

The network has to be accurate *where the economy actually spends its time*. So the
evaluation set is not a grid: it is drawn from a long simulation of the economy under the
current solution, which visits the ergodic distribution of $(z, g)$ states.

`build_ergodic_dataloaders` simulates `SIM_PATHS` economies for `SIM_T` steps, discards a
burn-in, and pools the resulting $(x, y, z, g)$ points into train and evaluation sets.

In [ ]:
t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    res = ct.build_ergodic_dataloaders(
        pinn_S, N_paths=SIM_PATHS, T=SIM_T, record_interval=2,
        batch_size_train=512, num_workers=0, seed=0,
    )
train_loader, eval_loader = res["train_loader"], res["eval_loader"]
print(f"ergodic simulation: {time.monotonic() - t0:.1f}s")
print(f"  train pool {len(train_loader.dataset):,} states | "
      f"eval pool {len(eval_loader.dataset):,} states")

## How well is the master equation satisfied?

The loss is the residual of the master equation for $S$ — the analogue of a Bellman
residual for a problem whose state includes an entire distribution. Averaging it by
$(x, y)$ shows *where* in type space the solution is weakest, which a single scalar loss
hides.

The absolute level is hard to read on its own — it carries the units of $S$, and it depends
on the evaluation pool: at `smoke` the short burn-in still includes states some distance
from the ergodic set, where the residual is naturally larger. Notebook 03 puts the level in
context by training a fresh network on the same pool and comparing.

In [ ]:
S_eval_list = plotting.freeze_S_batches(eval_loader, seed=1234)
maps_eval = plotting.compute_statewise_loss_maps_from_Slist(S_eval_list, ct, pinn_S)
E_all, extent = maps_eval["E_all"], maps_eval["extent"]

fig, ax = plt.subplots(1, 1, figsize=(5.5, 4.4), constrained_layout=True, dpi=140)
im = ax.imshow(np.sqrt(E_all.T), origin="lower", aspect="auto", extent=extent)
ax.set_title("Master-equation residual (RMSE)", fontsize=13)
ax.set_xlabel("worker type $x_i$", fontsize=12)
ax.set_ylabel("firm type $y_j$", fontsize=12)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
             format=mticker.FormatStrFormatter("%.1e"))
plt.show()

rmse = float(np.sqrt(np.nanmean(E_all)))
print(f"overall RMSE of the master-equation residual: {rmse:.3e}")
print(f"worst cell: {np.sqrt(np.nanmax(E_all)):.3e}   best cell: {np.sqrt(np.nanmin(E_all)):.3e}")
assert np.isfinite(rmse) and rmse < 1.0, (
    f"residual {rmse:.3e} is not a converged solution -- is the checkpoint loaded?"
)

## Summary

* The state of this economy is $(z, g)$ — an aggregate shock and a 55-dimensional
  distribution of matches. There is no way to put that on a grid.
* DeepSAM learns the surplus $S(x, y, z, g)$ as a neural network that takes $g$ as a direct
  input, and trains it on the residual of the master equation.
* The training points come from *simulating the model*, so accuracy is concentrated where
  the economy actually goes.

## Takeaway

The residual map is the diagnostic worth internalising. With no closed form and no coarse
benchmark to compare against, the check on the solution is that the equation it is supposed
to satisfy holds — everywhere the economy visits, not just on average.

Notebook 02 puts the solved model to work on the COVID experiment.